# CRF Feature Engineering — Concepts from Scratch
This notebook teaches every concept used in `word2features` step by step.

## 1. What Problem Are We Solving?

Given a recipe ingredient list like:
```
2 tablespoons Olive Oil
```
We want to label every word:
```
2            → quantity
tablespoons  → unit
Olive        → ingredient
Oil          → ingredient
```
This is called **Named Entity Recognition (NER)** — tagging each token with a class.

## 2. Why Not Just Use a Dictionary Lookup?

The word `cup` could be:
- **unit** → `1 cup flour`
- **ingredient** → `cup cake mix`

Context matters. We need a model that looks at **surrounding words**, not just the word itself.

## 3. What is a CRF?

**Conditional Random Field (CRF)** is a sequence labeling model.

- It labels every token in a sequence
- It considers the **entire sequence** when predicting each label
- It uses **features** (hand-crafted properties of tokens) to learn patterns

Unlike a simple classifier that predicts one token at a time, CRF says:
> *"If the previous label was `quantity`, the current token is likely `unit`"*

This is why it works well for NER.

## 4. What Are Features?

A CRF does not understand raw text. You must convert each token into a **dictionary of properties**.

```python
token = 'tablespoons'

features = {
    'token': 'tablespoons',
    'is_unit': True,
    'prev_is_digit': True,
    ...
}
```

The CRF learns weights for each feature — which features predict which label.

In [ ]:
# Setup
import re
import spacy

nlp = spacy.load('en_core_web_sm')
#          Model      │  Size   │ Accuracy │       Use when        │
#   ├─────────────────┼─────────┼──────────┼───────────────────────┤
#   │ en_core_web_sm  │ ~12 MB  │ lower    │ fast, low memory      │
#   ├─────────────────┼─────────┼──────────┼───────────────────────┤
#   │ en_core_web_md  │ ~43 MB  │ medium   │ word vectors included │                                                                                                                                              
#   ├─────────────────┼─────────┼──────────┼───────────────────────┤                                                                                                                                              
#   │ en_core_web_lg  │ ~741 MB │ higher   │ best accuracy         │                                                                                                                                              
#   ├─────────────────┼─────────┼──────────┼───────────────────────┤                                                                                                                                              
#   │ en_core_web_trf │ ~400 MB │ highest  │ transformer-based     |  

# Sample sentence we'll use throughout
sent = ['2', 'tablespoons', 'Olive', 'Oil']
print(sent)

['2', 'tablespoons', 'Olive', 'Oil']


## 5. Core Token Features

The simplest features describe the token itself — its shape, case, characters.

In [2]:
word = 'tablespoons'

print('lowercase     :', word.lower())
print('is_title      :', word.istitle())   # starts with capital
print('is_upper      :', word.isupper())   # ALL CAPS
print('has_digit     :', any(c.isdigit() for c in word))
print('has_alpha     :', any(c.isalpha() for c in word))
print('hyphenated    :', '-' in word)
print('slash_present :', '/' in word)

lowercase     : tablespoons
is_title      : False
is_upper      : False
has_digit     : False
has_alpha     : True
hyphenated    : False
slash_present : False


In [3]:
# Try with different tokens to see the difference
for word in ['2', '1/2', '1-1/2', 'Onion', 'OIL', 'tablespoon']:
    print(f"{word:15} | is_title={word.istitle()} | has_digit={any(c.isdigit() for c in word)} | hyphen={'-' in word} | slash={'/' in word}")

2               | is_title=False | has_digit=True | hyphen=False | slash=False
1/2             | is_title=False | has_digit=True | hyphen=False | slash=True
1-1/2           | is_title=False | has_digit=True | hyphen=True | slash=True
Onion           | is_title=True | has_digit=False | hyphen=False | slash=False
OIL             | is_title=False | has_digit=False | hyphen=False | slash=False
tablespoon      | is_title=False | has_digit=False | hyphen=False | slash=False


## 6. spaCy Linguistic Features

spaCy gives richer features — lemma, POS tag, dependency, shape — by parsing the sentence.

In [11]:
doc = nlp(' '.join(sent))
#  sent = ['2', 'tablespoons', 'Olive', 'Oil']                                                                                                                                                                   
#   ' '.join(sent)  →  '2 tablespoons Olive Oil'                                                                                                                                                                  
                                                                                                                                                                                                                
#   nlp(...) then parses that string and returns a spaCy Doc object:                                                                                                                                              
                  
#   doc = nlp('2 tablespoons Olive Oil')                                                                                                                                                                          
#   type(doc)  # <class 'spacy.tokens.doc.Doc'>
                                                                                                                                                                                                                
#   A Doc is like a list of spaCy Token objects — each token has all the linguistic properties attached:                                                                                                          
                                                                                                                                                                                                                
#   doc[0]        # spaCy Token for '2'                                                                                                                                                                           
#   doc[0].text   # '2'                                                                                                                                                                                           
#   doc[0].pos_   # 'NUM'
#   doc[0].lemma_ # '2'                                                                                                                                                                                           
                                                                                                                                                                                                                
#   doc[1]        # spaCy Token for 'tablespoons'
#   doc[1].pos_   # 'NOUN'                                                                                                                                                                                        
#   doc[1].lemma_ # 'tablespoon'
                                    

for tok in doc:
    print(f"{tok.text:15} lemma={tok.lemma_:15} pos={tok.pos_:8} tag={tok.tag_:6} dep={tok.dep_:10} shape={tok.shape_:8} stop={tok.is_stop}")

2               lemma=2               pos=NUM      tag=CD     dep=nummod     shape=d        stop=False
tablespoons     lemma=tablespoon      pos=NOUN     tag=NNS    dep=ROOT       shape=xxxx     stop=False
Olive           lemma=Olive           pos=PROPN    tag=NNP    dep=compound   shape=Xxxxx    stop=False
Oil             lemma=Oil             pos=PROPN    tag=NNP    dep=appos      shape=Xxx      stop=False


### What do these mean?

| Feature | Meaning | Example |
|---|---|---|
| `lemma` | base form of word | `tablespoons` → `tablespoon` |
| `pos_` | coarse POS tag | `NOUN`, `NUM`, `VERB` |
| `tag_` | fine-grained POS | `NNS` (plural noun), `CD` (cardinal number) |
| `dep_` | dependency relation | `nsubj`, `dobj`, `ROOT` |
| `shape_` | character shape | `Xxxxx`, `dd`, `x` |
| `is_stop` | is it a stopword | `True` for 'the', 'a', 'of' |

## 7. Regex Quantity Pattern

Quantities appear in many forms — we use regex to match all of them.

In [12]:
quantity_pattern = re.compile(
    r'^\d+$'          # whole number:    2, 500
    r'|^\d+\.\d+$'    # decimal:         3.14
    r'|^\d+/\d+$'     # fraction:        1/2
    r'|^\d+-\d+/\d+$' # mixed fraction:  1-1/2
)

test_tokens = ['2', '1/2', '1-1/2', '3.14', 'tablespoon', 'Onion', '500']

for t in test_tokens:
    print(f"{t:15} → is_quantity: {bool(quantity_pattern.match(t))}")

2               → is_quantity: True
1/2             → is_quantity: True
1-1/2           → is_quantity: True
3.14            → is_quantity: True
tablespoon      → is_quantity: False
Onion           → is_quantity: False
500             → is_quantity: True


## 8. Unit and Quantity Keywords

Some units and quantity words don't match a number pattern — we use keyword sets.

In [13]:
unit_keywords = {
    'cup', 'cups', 'tablespoon', 'tablespoons', 'tbsp', 'teaspoon',
    'teaspoons', 'tsp', 'gram', 'grams', 'kg', 'ml', 'liter', 'liters',
    'pinch', 'handful', 'bunch', 'clove', 'cloves', 'sprig', 'sprigs',
    'inch', 'inches', 'piece', 'pieces', 'slice', 'slices', 'dash'
}

quantity_keywords = {
    'half', 'quarter', 'third', 'double', 'triple', 'few', 'some',
    'several', 'little', 'whole'
}

test_tokens = ['cup', 'half', 'tablespoon', 'Onion', 'few', 'Oil']

for t in test_tokens:
    print(f"{t:15} → is_unit={t.lower() in unit_keywords} | is_quantity_kw={t.lower() in quantity_keywords}")

cup             → is_unit=True | is_quantity_kw=False
half            → is_unit=False | is_quantity_kw=True
tablespoon      → is_unit=True | is_quantity_kw=False
Onion           → is_unit=False | is_quantity_kw=False
few             → is_unit=False | is_quantity_kw=True
Oil             → is_unit=False | is_quantity_kw=False


## 9. Context Features — Why Neighbours Matter

The same word can have different labels depending on what surrounds it.

In [14]:
# 'cup' as unit vs ingredient
sent_unit = ['1', 'cup', 'flour']
sent_ingredient = ['cup', 'cake', 'mix']

[sent_unit, sent_ingredient]

for sent in [sent_unit, sent_ingredient]:
    i = sent.index('cup')
    prev = sent[i-1] if i > 0 else None
    nxt  = sent[i+1] if i < len(sent)-1 else None
    prev_is_digit = prev.isdigit() if prev else False
    print(f"sent={sent} | prev={prev} | prev_is_digit={prev_is_digit} | next={nxt}")

sent=['1', 'cup', 'flour'] | prev=1 | prev_is_digit=True | next=flour
sent=['cup', 'cake', 'mix'] | prev=None | prev_is_digit=False | next=cake


## 10. BOS and EOS — Sequence Boundaries

**BOS** (Beginning of Sentence) and **EOS** (End of Sentence) are special flags.

- First token in a recipe is often a **quantity** (`2 cups ...`)
- Last token is often an **ingredient** (`... Olive Oil`)

These position flags help the CRF learn such patterns.

In [15]:
sent = ['2', 'tablespoons', 'Olive', 'Oil']

for i, word in enumerate(sent):
    bos = (i == 0)
    eos = (i == len(sent) - 1)
    print(f"[{i}] {word:15} BOS={bos} | EOS={eos}")

[0] 2               BOS=True | EOS=False
[1] tablespoons     BOS=False | EOS=False
[2] Olive           BOS=False | EOS=False
[3] Oil             BOS=False | EOS=True


## 11. Putting It All Together — word2features

Now we combine everything into one function that returns a feature dictionary for token at position `i`.

In [16]:
def word2features(sent, i):
    word = sent[i]
    doc = nlp(' '.join(sent))
    tok = doc[i]

    features = {
        'bias': 1.0,
        'token': word.lower(),
        'lemma': tok.lemma_.lower(),
        'pos_tag': tok.pos_,
        'tag': tok.tag_,
        'dep': tok.dep_,
        'shape': tok.shape_,
        'is_stop': tok.is_stop,
        'is_digit': tok.is_digit,
        'has_digit': any(c.isdigit() for c in word),
        'has_alpha': any(c.isalpha() for c in word),
        'hyphenated': '-' in word,
        'slash_present': '/' in word,
        'is_title': word.istitle(),
        'is_upper': word.isupper(),
        'is_punct': tok.is_punct,
        'is_quantity': bool(quantity_pattern.match(word)) or word.lower() in quantity_keywords,
        'is_unit': word.lower() in unit_keywords,
        'is_numeric': bool(quantity_pattern.match(word)),
        'is_fraction': bool(re.match(r'^\d+/\d+$', word)),
        'is_decimal': bool(re.match(r'^\d+\.\d+$', word)),
        'preceding_word': sent[i-1].lower() if i > 0 else '',
        'following_word': sent[i+1].lower() if i < len(sent)-1 else '',
    }

    if i > 0:
        prev = sent[i-1]
        features['prev_token'] = prev.lower()
        features['prev_is_quantity'] = bool(quantity_pattern.match(prev)) or prev.lower() in quantity_keywords
        features['prev_is_digit'] = prev.isdigit()
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        nxt = sent[i+1]
        features['next_token'] = nxt.lower()
        features['next_is_unit'] = nxt.lower() in unit_keywords
        features['next_is_ingredient'] = nxt.lower() not in unit_keywords and not bool(quantity_pattern.match(nxt))
    else:
        features['EOS'] = True

    return features

In [17]:
# Inspect features for each token in a sentence
sent = ['2', 'tablespoons', 'Olive', 'Oil']

for i, word in enumerate(sent):
    print(f"\n--- Token: '{word}' ---")
    for k, v in word2features(sent, i).items():
        print(f"  {k:25} : {v}")


--- Token: '2' ---
  bias                      : 1.0
  token                     : 2
  lemma                     : 2
  pos_tag                   : NUM
  tag                       : CD
  dep                       : nummod
  shape                     : d
  is_stop                   : False
  is_digit                  : True
  has_digit                 : True
  has_alpha                 : False
  hyphenated                : False
  slash_present             : False
  is_title                  : False
  is_upper                  : False
  is_punct                  : False
  is_quantity               : True
  is_unit                   : False
  is_numeric                : True
  is_fraction               : False
  is_decimal                : False
  preceding_word            : 
  following_word            : tablespoons
  BOS                       : True
  next_token                : tablespoons
  next_is_unit              : True
  next_is_ingredient        : False

--- Token: 'tablespoons'

## 12. sent2features — Features for a Full Recipe

`word2features` handles one token. `sent2features` applies it to every token in a recipe.

In [ ]:
def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

# Each recipe → list of feature dicts (one per token)
features = sent2features(sent)
print(f"Recipe has {len(sent)} tokens → {len(features)} feature dicts")
print(f"\nFirst token features: {features[0]}")

## 13. The Full Pipeline

```
raw recipe string
       ↓  str.split()
list of tokens:  ['2', 'tablespoons', 'Olive', 'Oil']
       ↓  sent2features()
list of feature dicts:  [{...}, {...}, {...}, {...}]
       ↓  CRF.predict()
list of labels:  ['quantity', 'unit', 'ingredient', 'ingredient']
```

The CRF model learns which feature patterns → which labels during training.